# 🧠 Aula 03b: Normalização e Padronização de Características

Nesta aula prática, estudaremos um dos passos mais importantes do pré-processamento de dados em Machine Learning: a **Escala de Características** (Feature Scaling).

Muitos algoritmos (como o KNN, SVM e Gradiente Descendente) calculam distâncias ou direções de descida. Se uma característica tem escala em milhares (como `area`) e outra em decimais (como `smoothness`), o algoritmo dará peso desproporcional à característica de maior magnitude, ignorando as outras.

Aprenderemos a corrigir isso usando duas técnicas principais:
1. **Normalização (Min-Max Scaling):** Redimensiona os dados para o intervalo $[0, 1]$.
2. **Padronização (Standardization):** Transforma os dados para ter média $0$ e desvio padrão $1$ (escore Z).

⚠️ **Importante:** Discutiremos também o conceito crítico de **Vazamento de Dados (Data Leakage)** ao ajustar nossos scalers.

<a href="https://colab.research.google.com/github/Paulo83-dev/mestrado-computacao-aplicada/blob/main/machine-learning/aula03b - normalização.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
# Carrega a base e divide em treino e teste antes de qualquer escala
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### 🤖 KNN nos Dados Brutos (Sem Escala)

#### 🔍 O que este bloco faz?
Treina o modelo `KNeighborsClassifier` com os dados brutos e mede a acurácia.

#### 🎯 Qual a intenção pedagógica?
Veja que a acurácia é boa (~$92.98\%$), mas podemos fazer melhor. Como o KNN calcula a distância Euclidiana:
$$d(p, q) = \sqrt{\sum (p_i - q_i)^2}$$
As colunas com valores na casa dos milhares (como `area`) dominam completamente a distância, silenciando variáveis cruciais como `smoothness`.

In [90]:
from sklearn.neighbors import KNeighborsClassifier

# Treina o KNN com dados puros (sem normalização)
modelo = KNeighborsClassifier()
modelo.fit(X_train, y_train)
print("Acurácia sem escala:", modelo.score(X_test, y_test))

0.9298245614035088

### 📊 Analisando a Disparidade de Escalas

#### 🔍 O que este bloco faz?
Gera estatísticas descritivas (média, mínimo, máximo, desvio padrão) das características.

#### 🎯 Qual a intenção pedagógica?
Compare os valores máximos das colunas:
- `mean area`: máximo de `2501.0`
- `mean smoothness`: máximo de `0.1634`

Uma diferença de mais de 15.000 vezes! Fica evidente por que a normalização é necessária.

In [91]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Cria DataFrame para analisar as estatísticas descritivas
cancer_data = load_breast_cancer()
df_X = pd.DataFrame(X, columns=cancer_data.feature_names)
df_X.describe()

       mean radius  mean texture  mean perimeter    mean area  \
count   569.000000    569.000000      569.000000   569.000000   
mean     14.127292     19.289649       91.969033   654.889104   
std       3.524049      4.301036       24.298981   351.914129   
min       6.981000      9.710000       43.790000   143.500000   
25%      11.700000     16.170000       75.170000   420.300000   
50%      13.370000     18.840000       86.240000   551.100000   
75%      15.780000     21.800000      104.100000   782.700000   
max      28.110000     39.280000      188.500000  2501.000000   

       mean smoothness  mean compactness  mean concavity  mean concave points  \
count       569.000000        569.000000      569.000000           569.000000   
mean          0.096360          0.104341        0.088799             0.048919   
std           0.014064          0.052813        0.079720             0.038803   
min           0.052630          0.019380        0.000000             0.000000   
25%      

### 📏 Normalização Min-Max (Intervalo $[0, 1]$)

#### 🔍 O que este bloco faz?
Aplica a fórmula da normalização manualmente usando Pandas:
$$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

#### 🎯 Qual a intenção pedagógica?
Ao subtrair o mínimo e dividir pela amplitude (máximo - mínimo), forçamos todos os valores de todas as colunas a estarem contidos estritamente entre $0.0$ e $1.0$ (veja que no `describe` todos os mínimos agora são `0` e máximos são `1`).

In [92]:
maximo = df_X.max()
minimo = df_X.min()

# Normalização Min-Max manual
df_X_normalizado = (df_X - minimo) / (maximo - minimo)
df_X_normalizado.describe()

       mean radius  mean texture  mean perimeter   mean area  mean smoothness  \
count   569.000000    569.000000      569.000000  569.000000       569.000000   
mean      0.338222      0.323965        0.332935    0.216920         0.394785   
std       0.166787      0.145453        0.167915    0.149274         0.126967   
min       0.000000      0.000000        0.000000    0.000000         0.000000   
25%       0.223342      0.218465        0.216847    0.117413         0.304595   
50%       0.302381      0.308759        0.293345    0.172895         0.390358   
75%       0.416442      0.408860        0.416765    0.271135         0.475490   
max       1.000000      1.000000        1.000000    1.000000         1.000000   

       mean compactness  mean concavity  mean concave points  mean symmetry  \
count        569.000000      569.000000           569.000000     569.000000   
mean           0.260601        0.208058             0.243137       0.379605   
std            0.161992        0.

### 🛠️ Usando MinMaxScaler do Scikit-Learn

#### 🔍 O que este bloco faz?
Aplica a normalização usando a ferramenta oficial `MinMaxScaler` do Scikit-Learn.

#### 🎯 Qual a intenção pedagógica?
O Scikit-Learn abstrai o cálculo facilitando a integração em pipelines de produção. Note que o resultado final das médias é exatamente igual ao calculado manualmente.

In [93]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Normalização utilizando MinMaxScaler do Sklearn
scaler = MinMaxScaler()
df_X_normalizado = scaler.fit_transform(df_X)
print("Média das colunas normalizadas:\n", np.mean(df_X_normalizado, axis=0))

[0.33822196 0.32396512 0.33293507 0.21692009 0.39478452 0.26060053
 0.20805838 0.24313691 0.37960537 0.27037931 0.10634512 0.18932404
 0.09937611 0.06263579 0.18111904 0.17443851 0.08053969 0.22345401
 0.17814345 0.10019291 0.29666275 0.36399849 0.28313767 0.1709062
 0.40413785 0.22021232 0.21740294 0.39383582 0.26330686 0.18959607]


### ⚠️ Prevenindo Vazamento de Dados (Data Leakage)

#### 🔍 O que este bloco faz?
Aplica a normalização separando treino e teste, usando os parâmetros mínimo/máximo obtidos **apenas** do conjunto de treino para normalizar ambos os conjuntos.

#### 🎯 Qual a intenção pedagógica?
Este é um dos conceitos mais importantes em Machine Learning:
- **Regra de Ouro:** O conjunto de teste representa dados futuros, nunca vistos. Nós **nunca** podemos espiar o conjunto de teste para calcular médias, mínimos ou máximos.
- Se normalizarmos o dataset inteiro de uma vez antes do split, a informação do teste "vaza" para o treino (Data Leakage), inflando artificialmente nossa avaliação de desempenho.
- Por isso, calculamos `minimo_treino` e `maximo_treino` com `X_train` e usamos esses mesmos parâmetros para ajustar o `X_test`.
- Veja que ao normalizar corretamente, o desempenho do KNN salta de $92.9\%$ para **$96.49\%$**!

In [94]:
maximo_treino = X_train.max(axis=0)
minimo_treino = X_train.min(axis=0)

# Normaliza treino e teste usando as estatísticas do treino (sem vazamento de dados)
X_train_normalizado = (X_train - minimo_treino) / (maximo_treino - minimo_treino)
X_test_normalizado = (X_test - minimo_treino) / (maximo_treino - minimo_treino)

modelo = KNeighborsClassifier()
modelo.fit(X_train_normalizado, y_train)
print("Acurácia com Normalização Min-Max:", modelo.score(X_test_normalizado, y_test))

0.9649122807017544

### ⚖️ Padronização (Standardization / Z-Score)

#### 🔍 O que este bloco faz?
Aplica a padronização manual:
$$X_{std} = \frac{X - \mu}{\sigma}$$
Onde $\mu$ é a média e $\sigma$ é o desvio padrão.

#### 🎯 Qual a intenção pedagógica?
A padronização centraliza os dados na média zero e os escala para ter variância unitária (desvio padrão 1). Ela é menos sensível a *outliers* do que a normalização Min-Max, pois não tem limites fixos ($0$ e $1$).

In [95]:
media_treino = X_train.mean(axis=0)
desvio_treino = X_train.std(axis=0)

# Padronização manual utilizando média e desvio padrão do treino
X_train_padronizado = (X_train - media_treino) / desvio_treino
X_test_padronizado = (X_test - media_treino) / desvio_treino
print("Média das colunas de treino padronizadas (deve ser ~0):\n", np.mean(X_train_padronizado, axis=0))

[ 1.20355496e-16 -3.98216259e-15 -3.30565856e-16  1.20221293e-15
 -8.18392972e-16 -2.16261685e-15 -2.85729926e-16  9.29903285e-16
 -2.95636531e-15  3.60297872e-15  1.65533033e-15  2.04159034e-15
 -2.64989496e-16  4.22006752e-16 -1.57334463e-15 -6.21968899e-16
  1.13218348e-16 -1.70803542e-18 -3.91286515e-15  7.13714802e-16
 -1.65435431e-15  2.97045660e-15  6.60887706e-16 -9.69188100e-16
  6.07304195e-15 -9.88220494e-17  9.22339128e-16  2.83533880e-16
 -4.19688704e-15  7.78864153e-16]


### 🤖 KNN nos Dados Padronizados

#### 🔍 O que este bloco faz?
Treina e avalia o KNN usando os dados escalados pela padronização manual.

#### 🎯 Qual a intenção pedagógica?
Observe que a acurácia também atinge o patamar ideal de **$96.49\%$**, mostrando que ambas as técnicas de escala resolvem o problema do desbalanço dimensional no cálculo de distâncias do KNN.

In [96]:
modelo = KNeighborsClassifier()
modelo.fit(X_train_padronizado, y_train)
print("Acurácia com Padronização:", modelo.score(X_test_padronizado, y_test))

0.9649122807017544

### 🛠️ Usando StandardScaler do Scikit-Learn

#### 🔍 O que este bloco faz?
Aplica a padronização oficial com `StandardScaler`. Faz o `fit_transform` nos dados de treino (calcula média/desvio e aplica) e apenas `transform` nos dados de teste (aplica as mesmas estatísticas de treino no teste).

#### 🎯 Qual a intenção pedagógica?
A dobradinha `fit_transform(X_train)` e `transform(X_test)` é o padrão da indústria no Scikit-Learn para garantir que não haja vazamento de dados de forma simples e robusta.

In [98]:
from sklearn.preprocessing import StandardScaler

# Padronização com StandardScaler do Sklearn
scaler = StandardScaler()
X_train_padronizado = scaler.fit_transform(X_train)
X_test_padronizado = scaler.transform(X_test)

modelo = KNeighborsClassifier()
modelo.fit(X_train_padronizado, y_train)
print("Acurácia final com StandardScaler:", modelo.score(X_test_padronizado, y_test))

0.9649122807017544